In [1]:
# ===== CELL 1: SETUP =====
import pandas as pd
import numpy as np
import os
import re
import sqlite3

RAW_DIR = "data/raw"
DB_PATH = "data/startup_intelligence.db"

conn = sqlite3.connect(DB_PATH)

print("Setup complete. Connected to:", DB_PATH)

Setup complete. Connected to: data/startup_intelligence.db


In [2]:
df_openvc = pd.read_csv(os.path.join(RAW_DIR, "openvc_investors.csv"))
print(df_openvc.shape)
df_openvc[['name', 'countries', 'check_size', 'stages', 'thesis']].head()

(200, 12)


,name,countries,check_size,stages,thesis
0,Tunitas Ventures,USA,$500k to $1M,3. Early Revenue,We invest in category creating companies acros...
1,Maniv Mobility,"Brazil, Canada",$1M to $5M,"1. Idea or Patent, 2. Prototype",We invest in the world's leading mobility star...
2,Sentiero Ventures,"Canada, USA",$200k to $500k,"2. Prototype, 3. Early Revenue",We invest in B2B AI-enabled SaaS startups that...
3,Advantary Capital,USA,$2M to $10M,"3. Early Revenue, 4. Scaling","We invest in Seed through growth, sector agnos..."
4,Curiosity VC,"Belgium, Denmark",$500k to $1M,"3. Early Revenue, 4. Scaling",We invest in B2B software with a focus on appl...


In [3]:
def parse_check_size(val):
    """Extract a numeric check size in $ from messy strings like '\n1M' or '500k'"""
    if pd.isna(val):
        return None
    val = str(val).strip().upper()
    match = re.search(r'([\d.]+)\s*([KM])', val)
    if not match:
        return None
    num, unit = match.groups()
    num = float(num)
    return num * 1000 if unit == 'K' else num * 1_000_000

def parse_stages(val):
    """Split multi-stage strings into a clean list, stripping the leading numbers"""
    if pd.isna(val):
        return []
    stages = [s.strip() for s in str(val).split(',')]
    # strip leading "1. ", "2. " etc.
    stages = [re.sub(r'^\d+\.\s*', '', s) for s in stages]
    return stages

df_openvc['check_size_usd'] = df_openvc['check_size'].apply(parse_check_size)
df_openvc['stages_list'] = df_openvc['stages'].apply(parse_stages)

df_openvc[['name', 'check_size', 'check_size_usd', 'stages', 'stages_list']].head()

,name,check_size,check_size_usd,stages,stages_list
0,Tunitas Ventures,$500k to $1M,500000.0,3. Early Revenue,[Early Revenue]
1,Maniv Mobility,$1M to $5M,1000000.0,"1. Idea or Patent, 2. Prototype","[Idea or Patent, Prototype]"
2,Sentiero Ventures,$200k to $500k,200000.0,"2. Prototype, 3. Early Revenue","[Prototype, Early Revenue]"
3,Advantary Capital,$2M to $10M,2000000.0,"3. Early Revenue, 4. Scaling","[Early Revenue, Scaling]"
4,Curiosity VC,$500k to $1M,500000.0,"3. Early Revenue, 4. Scaling","[Early Revenue, Scaling]"


In [6]:
def compute_investor_fit(startup_sector, startup_stage, startup_funding_needed_usd, investors_df):
    """
    Score every investor's fit for a given startup profile.
    Returns a ranked DataFrame with match scores and reasons.
    """
    results = []
    
    for _, inv in investors_df.iterrows():
        score = 0
        reasons = []
        
        # Stage match (40 points) - does the investor invest at this stage?
        if startup_stage in inv['stages_list']:
            score += 40
            reasons.append(f"Invests at {startup_stage} stage")
        
        # Thesis/sector match (35 points) - simple keyword check against their thesis text
        thesis = str(inv['thesis']).lower()
        if pd.notna(inv['thesis']) and startup_sector.lower() in thesis:
            score += 35
            reasons.append(f"Thesis mentions {startup_sector}")
        
        # Check size match (25 points) - is the startup's ask within their typical range
        if pd.notna(inv['check_size_usd']) and startup_funding_needed_usd:
            # within 3x range either direction = reasonable fit
            ratio = startup_funding_needed_usd / inv['check_size_usd']
            if 0.33 <= ratio <= 3:
                score += 25
                reasons.append(f"Check size range aligns (~${inv['check_size_usd']:,.0f})")
        
        results.append({
            'investor_name': inv['name'],
            'firm_type': inv['firm_type'],
            'fit_score': score,
            'match_reasons': '; '.join(reasons) if reasons else 'No strong match signals',
            'countries': inv['countries']
        })
    
    return pd.DataFrame(results).sort_values('fit_score', ascending=False)

In [7]:
# Pull a real startup example from SQL to test against
test_startup = pd.read_sql_query(
    "SELECT startup_name, sector FROM dim_startup LIMIT 1", conn
).iloc[0]

matches = compute_investor_fit(
    startup_sector=test_startup['sector'],
    startup_stage='Early Revenue',  # example stage
    startup_funding_needed_usd=1_000_000,
    investors_df=df_openvc
)

print(f"Top investor matches for {test_startup['startup_name']} ({test_startup['sector']}):")
matches.head(10)

Top investor matches for DataTech (Gaming & Entertainment):


,investor_name,firm_type,fit_score,match_reasons,countries
0,Tunitas Ventures,VC firm,65,Invests at Early Revenue stage; Check size ran...,USA
29,Nordic Eye Venture...,VC firm,65,Invests at Early Revenue stage; Check size ran...,"Denmark, El Salvador"
32,Dimension,VC firm,65,Invests at Early Revenue stage; Check size ran...,USA
168,She Capital,VC firm,65,Invests at Early Revenue stage; Check size ran...,India
38,Venga Ventures,VC firm,65,Invests at Early Revenue stage; Check size ran...,"Austria, Germany"
40,Storm Ventures,VC firm,65,Invests at Early Revenue stage; Check size ran...,"USA, South Korea"
41,Initialized Capita...,VC firm,65,Invests at Early Revenue stage; Check size ran...,"Canada, Estonia"
43,Voyager Ventures,VC firm,65,Invests at Early Revenue stage; Check size ran...,"Denmark, UK"
160,Navivo Capital,VC firm,65,Invests at Early Revenue stage; Check size ran...,"Germany, France"
101,Electric Capital,VC firm,65,Invests at Early Revenue stage; Check size ran...,"Canada, India"


In [9]:
SECTOR_KEYWORDS = {
    'Gaming & Entertainment': ['gaming', 'game', 'entertainment', 'media', 'interactive'],
    'Artificial Intelligence': ['ai', 'artificial intelligence', 'machine learning', 'ml'],
    'Fintech': ['fintech', 'financial', 'banking', 'payments'],
    'Healthcare & Biotech': ['health', 'healthcare', 'biotech', 'medical', 'life sciences'],
    'SaaS / Enterprise Software': ['saas', 'enterprise software', 'b2b software', 'enterprise'],
    'E-Commerce': ['e-commerce', 'ecommerce', 'commerce', 'retail', 'consumer'],
    'Cybersecurity': ['cybersecurity', 'security', 'infosec'],
    'Climate Tech / Clean Energy': ['climate', 'clean energy', 'sustainability', 'cleantech'],
    'Food & AgriTech': ['food', 'agritech', 'agriculture', 'agtech'],
    'Robotics & Automation': ['robotics', 'automation', 'hardware'],
    'Space Tech': ['space', 'aerospace'],
    'HR Tech': ['hr tech', 'human resources', 'talent', 'future of work'],
    'Logistics & Supply Chain': ['logistics', 'supply chain', 'shipping'],
    'Legal Tech': ['legal', 'legaltech', 'compliance'],
    'Real Estate Tech': ['real estate', 'proptech'],
    'Social Media / Creator Economy': ['social', 'creator economy', 'community', 'consumer'],
    'Insurtech': ['insurtech', 'insurance'],
    'Travel Tech': ['travel', 'hospitality'],
    'Edtech': ['edtech', 'education', 'learning'],
    'Web3 / Blockchain': ['web3', 'blockchain', 'crypto', 'defi']
}

def compute_investor_fit(startup_sector, startup_stage, startup_funding_needed_usd, investors_df):
    results = []
    keywords = SECTOR_KEYWORDS.get(startup_sector, [startup_sector.lower()])
    
    for _, inv in investors_df.iterrows():
        score = 0
        reasons = []
        
        if startup_stage in inv['stages_list']:
            score += 40
            reasons.append(f"Invests at {startup_stage} stage")
        
        thesis = str(inv['thesis']).lower() if pd.notna(inv['thesis']) else ''
        matched_kw = [kw for kw in keywords if kw in thesis]
        if matched_kw:
            score += 35
            reasons.append(f"Thesis matches: {', '.join(matched_kw)}")
        
        if pd.notna(inv['check_size_usd']) and startup_funding_needed_usd:
            ratio = startup_funding_needed_usd / inv['check_size_usd']
            if 0.33 <= ratio <= 3:
                score += 25
                reasons.append(f"Check size aligns (~${inv['check_size_usd']:,.0f})")
        
        results.append({
            'investor_name': inv['name'],
            'firm_type': inv['firm_type'],
            'fit_score': score,
            'match_reasons': '; '.join(reasons) if reasons else 'No strong match signals',
            'countries': inv['countries']
        })
    
    return pd.DataFrame(results).sort_values('fit_score', ascending=False)

In [10]:
matches = compute_investor_fit(
    startup_sector=test_startup['sector'],
    startup_stage='Early Revenue',
    startup_funding_needed_usd=1_000_000,
    investors_df=df_openvc
)
print(f"Top investor matches for {test_startup['startup_name']} ({test_startup['sector']}):")
matches.head(10)

Top investor matches for DataTech (Gaming & Entertainment):


,investor_name,firm_type,fit_score,match_reasons,countries
188,DOMiNO Ventures,VC firm,75,Invests at Early Revenue stage; Thesis matches...,"Azerbaijan, Kazakhstan"
0,Tunitas Ventures,VC firm,65,Invests at Early Revenue stage; Check size ali...,USA
32,Dimension,VC firm,65,Invests at Early Revenue stage; Check size ali...,USA
168,She Capital,VC firm,65,Invests at Early Revenue stage; Check size ali...,India
38,Venga Ventures,VC firm,65,Invests at Early Revenue stage; Check size ali...,"Austria, Germany"
40,Storm Ventures,VC firm,65,Invests at Early Revenue stage; Check size ali...,"USA, South Korea"
41,Initialized Capita...,VC firm,65,Invests at Early Revenue stage; Check size ali...,"Canada, Estonia"
43,Voyager Ventures,VC firm,65,Invests at Early Revenue stage; Check size ali...,"Denmark, UK"
160,Navivo Capital,VC firm,65,Invests at Early Revenue stage; Check size ali...,"Germany, France"
101,Electric Capital,VC firm,65,Invests at Early Revenue stage; Check size ali...,"Canada, India"


In [11]:
keywords = SECTOR_KEYWORDS.get('Gaming & Entertainment', [])
print(f"Keywords being checked: {keywords}")

# Count how many investors' thesis text actually contains any of these keywords
match_count = df_openvc['thesis'].apply(
    lambda t: any(kw in str(t).lower() for kw in keywords) if pd.notna(t) else False
).sum()
print(f"Investors with a thesis match: {match_count} out of {len(df_openvc)}")

# Look at a sample of thesis text to see what's actually in there
print(df_openvc['thesis'].dropna().sample(5, random_state=1).tolist()) 


Keywords being checked: ['gaming', 'game', 'entertainment', 'media', 'interactive']
Investors with a thesis match: 2 out of 200
["We invest in product-market fit stage b2b vertical saas companies with ARR of $2 mm+ and growing.  Typical companies are vertical SaaS systems of record, intelligence, engagement, marketplaces with founder /SME's that are looking to grow rapidly with less risk.  We like niche verticals and write $2 to $30 mm checks for growth and recap.  We are happy in syndicates and lead rounds and have been a partner and founder-friendly firm since our inception. Vertical B2B SaaS is our focus.", 'We invest in B2B software startups in areas such as SaaS, enterprise infrastructure, cybersecurity, artificial intelligence with $1M+ ARR.', 'We invest in cybersecurity companies at seed (ideation) and Series A ($1m+ ARR or on trajectory)', 'We invest in B2B SaaS solutions.', 'We invest in Pre-Seed up to Series A round. Sectors that we have invested in: E-Commerce, Marketplace, 

In [12]:
# Rebalance: stage (35) + check size (35) + thesis bonus (30) 
# Thesis becomes a differentiator/bonus rather than a required signal
def compute_investor_fit(startup_sector, startup_stage, startup_funding_needed_usd, investors_df):
    results = []
    keywords = SECTOR_KEYWORDS.get(startup_sector, [startup_sector.lower()])
    
    for _, inv in investors_df.iterrows():
        score = 0
        reasons = []
        
        if startup_stage in inv['stages_list']:
            score += 35
            reasons.append(f"Invests at {startup_stage} stage")
        
        if pd.notna(inv['check_size_usd']) and startup_funding_needed_usd:
            ratio = startup_funding_needed_usd / inv['check_size_usd']
            if 0.33 <= ratio <= 3:
                score += 35
                reasons.append(f"Check size aligns (~${inv['check_size_usd']:,.0f})")
        
        thesis = str(inv['thesis']).lower() if pd.notna(inv['thesis']) else ''
        matched_kw = [kw for kw in keywords if kw in thesis]
        if matched_kw:
            score += 30
            reasons.append(f"Thesis matches: {', '.join(matched_kw)} (specialist fit)")
        
        results.append({
            'investor_name': inv['name'],
            'firm_type': inv['firm_type'],
            'fit_score': score,
            'match_reasons': '; '.join(reasons) if reasons else 'No strong match signals',
            'countries': inv['countries']
        })
    
    return pd.DataFrame(results).sort_values('fit_score', ascending=False)

In [13]:
matches_gaming = compute_investor_fit('Gaming & Entertainment', 'Early Revenue', 1_000_000, df_openvc)
matches_saas = compute_investor_fit('SaaS / Enterprise Software', 'Early Revenue', 1_000_000, df_openvc)

print("Gaming & Entertainment top 5:")
print(matches_gaming.head(5))
print("\nSaaS / Enterprise Software top 5:")
print(matches_saas.head(5))

Gaming & Entertainment top 5:
        investor_name firm_type  fit_score  \
0    Tunitas Ventures   VC firm         70   
139           Capagro   VC firm         70   
32          Dimension   VC firm         70   
168       She Capital   VC firm         70   
38     Venga Ventures   VC firm         70   

                                         match_reasons         countries  
0    Invests at Early Revenue stage; Check size ali...               USA  
139  Invests at Early Revenue stage; Check size ali...    France, Sweden  
32   Invests at Early Revenue stage; Check size ali...               USA  
168  Invests at Early Revenue stage; Check size ali...             India  
38   Invests at Early Revenue stage; Check size ali...  Austria, Germany  

SaaS / Enterprise Software top 5:
             investor_name firm_type  fit_score  \
58         Frontier Growth   VC firm        100   
8             Step Venture   VC firm        100   
153  Scale Capital (Den...   VC firm        100   
160 

In [14]:
def get_investor_matches(startup_name, startup_sector, startup_stage, funding_needed_usd, top_n=10):
    """
    Returns top N investor matches for a given startup profile.
    
    Note: Thesis-keyword matching is sparse by design - the OpenVC dataset 
    skews heavily toward B2B/SaaS-focused investors, so consumer-facing 
    sectors (e.g. Gaming & Entertainment) will show fewer thesis matches. 
    This reflects a real characteristic of the investor pool, not a bug - 
    stage and check-size fit carry proportionally more weight for 
    underrepresented sectors.
    """
    matches = compute_investor_fit(startup_sector, startup_stage, funding_needed_usd, df_openvc)
    return matches.head(top_n)

# Quick test
get_investor_matches('DataTech', 'Gaming & Entertainment', 'Early Revenue', 1_000_000)

,investor_name,firm_type,fit_score,match_reasons,countries
0,Tunitas Ventures,VC firm,70,Invests at Early Revenue stage; Check size ali...,USA
139,Capagro,VC firm,70,Invests at Early Revenue stage; Check size ali...,"France, Sweden"
32,Dimension,VC firm,70,Invests at Early Revenue stage; Check size ali...,USA
168,She Capital,VC firm,70,Invests at Early Revenue stage; Check size ali...,India
38,Venga Ventures,VC firm,70,Invests at Early Revenue stage; Check size ali...,"Austria, Germany"
40,Storm Ventures,VC firm,70,Invests at Early Revenue stage; Check size ali...,"USA, South Korea"
41,Initialized Capita...,VC firm,70,Invests at Early Revenue stage; Check size ali...,"Canada, Estonia"
43,Voyager Ventures,VC firm,70,Invests at Early Revenue stage; Check size ali...,"Denmark, UK"
160,Navivo Capital,VC firm,70,Invests at Early Revenue stage; Check size ali...,"Germany, France"
58,Frontier Growth,VC firm,70,Invests at Early Revenue stage; Check size ali...,"USA, Canada"
